# importing libraries

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec

# loading dataset

In [3]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [4]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
df['label'].value_counts()

,count
label,
ham,4825
spam,747


In [6]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

In [7]:
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# vocabulary stats

In [8]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# model development

In [9]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(1,), dtype=tf.string))

In [10]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=20000,#totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=30#averageWordsLength,
)
text_vec.adapt(X_train)
cnn.add(text_vec)

In [11]:
vocab_size = text_vec.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))

In [13]:
import re
import string

# Function to mimic 'lower_and_strip_punctuation'
def standardize_text(text):
    text = text.lower()
    return re.sub(f'[{re.escape(string.punctuation)}]', '', text)

# [text_vec.standardize(s).numpy().decode().split()
#              for s in X_train]

sentences = [standardize_text(str(s)).split() for s in X_train]
embedding_dim = 128

w2v = Word2Vec(
    sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1
)

embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]

In [14]:
cnn.add(tf.keras.layers.Embedding(
    input_dim=len(vocab_size),
    output_dim=embedding_dim,
    input_length=averageWordsLength,
    weights=[embedding_matrix],
    trainable=False
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=2, activation='softmax'))

In [16]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 30)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 30, 128)        │     1,089,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 26, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187,842 (4.53 MB)

 Trainable params: 98,818 (386.01 KB)

 Non-trainable params: 1,089,024 (4.15 MB)

In [17]:
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['recall'])

# model training

In [18]:
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.4729 - recall: 0.8597 - val_loss: 0.2960 - val_recall: 0.8637
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2832 - recall: 0.8689 - val_loss: 0.3249 - val_recall: 0.8655
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3144 - recall: 0.8584 - val_loss: 0.2853 - val_recall: 0.8637
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2697 - recall: 0.8686 - val_loss: 0.2771 - val_recall: 0.8726
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2596 - recall: 0.8818 - val_loss: 0.2769 - val_recall: 0.8726


# model evaluation

In [19]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

# Convert the list of messages to a TensorFlow constant of strings
test_messages_tf = tf.constant(test_messages, dtype=tf.string)

preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    # print(np.argmax(p))
    label = "Ham" if np.argmax(p)==0 else "Spam"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
Ham → Hey, are we meeting today? (0.99)
Ham → WIN cash now!!! Click the link (0.98)
Ham → Your OTP is 348921 (0.99)
Ham → You have won $1000! (0.99)
Ham → Congratulations! You have won a free lottery ticket (0.99)
Ham → Don't forget to submit the assignment (0.98)
Ham → URGENT! You won a cash prize. Call immediately! (0.99)
Ham → My friend won a prize yesterday (0.99)


In [20]:
from sklearn.metrics import classification_report, f1_score
print(classification_report(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))
print(f1_score(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       966
           1       0.53      0.46      0.49       149

    accuracy                           0.87      1115
   macro avg       0.72      0.70      0.71      1115
weighted avg       0.87      0.87      0.87      1115

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
0.4892086330935252
